# torch.tensor

In [ ]:
import torch

# input 값(데이터)
x = torch.tensor([2.0], requires_grad=True)

# 가중치1
w = torch.tensor([3.0], requires_grad=True)

# 가중치2
b = torch.tensor([1.0], requires_grad=True)

y = w * x + b
loss = (y - 5.0) ** 2 # Loss Function

loss.backward() # dL/dw, dL/db 자동 계산
print(w.grad, b.grad)

# torch.nn & Module

In [ ]:
import torch.nn as nn

# 간단한 모델 생성
class TinyMLP(nn.Module):
    def __init__(self, in_dim=784, hidden=128, out_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden) # fully connection layer
        self.act1 = nn.ReLU() # 활성화 함수
        self.fc2 = nn.Linear(hidden, out_dim) # fully connection layer
        self.act2 = nn.Softmax() # 활성화 함수

    # 꼭 선언해줘야함.
    def forward(self, x):
        x = self.fc1(x)
        x = self.act1(x)
        x = self.fc2(x)
        x = self.act2(x)

        return x

In [ ]:
model = TinyMLP()

In [ ]:
model

In [ ]:
# 784차원 임의의 입력 데이터 생성
x = torch.randn(1, 784)

x.shape

In [ ]:
# forward가 알아서 작동
model(x)

# Dataset & DataLoader

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

tfm = transforms.Compose([transforms.ToTensor()])
train_set = datasets.MNIST(root="./data", train=True,
                           download=True, transform=tfm)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

images, labels = next(iter(train_loader))
print(images, labels)

In [ ]:
# batch size를 64로 설정하였으므로 64개의 데이터 반환
images.shape, labels.shape

# Model 학습

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# DataLoader는 iterator다.
for epoch in range(5):
    for images, labels in train_loader:
        images = images.view(images.size(0), -1) # (64, 784)로 shape 변형

        # 1. 모델 예측
        preds = model(images)
        # 2. 손실 계산
        loss = criterion(preds, labels)

        # 3. 이전 기울기 초기화
        optimizer.zero_grad()

        # 4. 역전파(backpropagation) -> 기울기 계산
        loss.backward()
        
        # 5. 파라미터 업데이트
        optimizer.step()

    print(f"epoch {epoch} loss {loss.item():.4f}")

# Model 평가

In [ ]:
model.eval()
correct, total = 0, 0

test_loader = DataLoader(train_set, batch_size=64, shuffle=False)

with torch.no_grad(): # 기울기 계산을 끔
    for images, labels in test_loader:
        images = images.view(images.size(0), -1)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0) # labels.shape[0]

print(f"accuracy: {correct/total:.2%}")
model.train() # 다시 학습 모드로 전환

# 전체 코드

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# GPU 사용 가능하면 GPU, 아니면 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 입력 크기가 고정일 때 GPU에서 최적의 conv 알고리즘을 캐싱해 속도를 높여줌
torch.backends.cudnn.benchmark = (device.type == "cuda")


# data 불러오기
# 하지만, 실제로는 아래처럼 정제되어 있지 않은 경우가 대부분이므로 따로 나눌 것.
train_set = datasets.MNIST(root="./data", train=True,
                           download=True, transform=tfm)

train_images = train_set.data # 학습 이미지
train_labels = train_set.targets # 학습 정답
train_classes = train_set.class_to_idx # 보통 label은 숫자로 나오므로 이를 문자형으로 바꾸기 위한 딕셔너리

# train과 test dataset으로 분할하기
train_images, test_images, train_labels, test_labels = train_test_split(train_images, train_labels, stratify=train_labels, test_size=0.2)

# pretrained model 불러오기 or Custom model 불러오기
model = resnet50(weights="IMAGENET1K_V1")
model.fc = nn.Sequential(nn.Linear(2048, 10), nn.Softmax(dim=1))
model = model.to(device)

# augmentation 생성
# 여기서는 pretrained model의 preprocess 메커니즘을 따름
weights = ResNet50_Weights.DEFAULT
transform = weights.transforms()

# Dataset 생성
class CustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return self.images.shape[0]

    def __getitem__(self, idx):
        batch_images = self.images[idx]
        batch_images = batch_images.expand(3, -1, -1)
        batch_labels = self.labels[idx]

        if self.transform:
            preprocessed_batch_images = self.transform(batch_images)
            return preprocessed_batch_images, batch_labels
        else:
            return batch_images, batch_labels

train_dataset = CustomDataset(train_images, train_labels, transform=transform)
test_dataset = CustomDataset(test_images, test_labels, transform=transform)

# DataLoader 생성
# pin_memory: GPU 사용 시 CPU->GPU 텐서 전송 속도를 높여줌
PIN_MEMORY = device.type == "cuda"
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=PIN_MEMORY)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=PIN_MEMORY)

# 손실함수 & optimizer 생성
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# tqdm 갱신 주기 (20 batch마다 한 번씩만 postfix 갱신)
LOG_INTERVAL = 20

# Train 및 Eval 로직
num_epochs = 5
for epoch in range(num_epochs):
    # 1. model train
    model.train()
    train_loss = 0.0
    train_preds_all, train_labels_all = [], []

    with tqdm(train_dataloader, desc=f"[Epoch {epoch+1}/{num_epochs}] Train", leave=False) as train_bar:
        for batch_idx, (images, labels) in enumerate(train_bar):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            # 1-1. model 예측
            preds = model(images)

            # 1-2. loss function 계산
            loss = criterion(preds, labels)

            # 1-3. 이전 기울기 초기화
            optimizer.zero_grad()

            # 1-4. 역전파(backpropagation) -> 기울기 계산
            loss.backward()

            # 1-5. 파라미터 업데이트
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            train_preds_all.extend(preds.argmax(dim=1).cpu().tolist())
            train_labels_all.extend(labels.cpu().tolist())

            # 20 batch마다 한 번씩 loss/accuracy 갱신
            if (batch_idx + 1) % LOG_INTERVAL == 0:
                train_accuracy = accuracy_score(train_labels_all, train_preds_all)
                train_bar.set_postfix(loss=f"{loss.item():.4f}", accuracy=f"{train_accuracy:.2%}")

    train_loss /= len(train_dataloader.dataset)
    train_accuracy = accuracy_score(train_labels_all, train_preds_all)

    # 2. model eval
    model.eval()
    eval_preds_all, eval_labels_all = [], []
    accuracy = 0.0

    with torch.no_grad(), tqdm(test_dataloader, desc=f"[Epoch {epoch+1}/{num_epochs}] Eval", leave=False) as eval_bar:
        for batch_idx, (images, labels) in enumerate(eval_bar):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            preds = model(images)
            eval_preds_all.extend(preds.argmax(dim=1).cpu().tolist())
            eval_labels_all.extend(labels.cpu().tolist())

            # 20 batch마다 한 번씩 accuracy 갱신
            if (batch_idx + 1) % LOG_INTERVAL == 0:
                accuracy = accuracy_score(eval_labels_all, eval_preds_all)
                eval_bar.set_postfix(accuracy=f"{accuracy:.2%}")

    accuracy = accuracy_score(eval_labels_all, eval_preds_all)

    print(f"epoch {epoch+1}/{num_epochs} | train_loss {train_loss:.4f} | train_accuracy {train_accuracy:.2%} | accuracy {accuracy:.2%}")

# 모델 저장
model_path = "resnet50_mnist.pth"
torch.save(model.state_dict(), model_path)
print(f"모델 저장 완료: {model_path}")